# LSTM Definition and Training
This section defines the baseline LSTM model. It consists of two classes:
* LSTMModel: This is the core nn.Module that defines the neural network architecture. It's a simple, single-layer LSTM followed by a fully-connected layer. The model takes a sequence of historical features for a stock and outputs a single value predicting the next day's return. Each stock is treated as an independent time series.
* LSTM: This is a wrapper class that manages the entire training and evaluation workflow for the LSTMModel. It handles loading the data, creating batches, running the training loop for a specified number of epochs, performing validation, and saving the best-performing model. The loss function used is a simple Mean Squared Error (MSE), which measures the squared difference between the predicted and actual returns.

In [1]:
import argparse
import copy
import numpy as np
import os
import sys
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from time import time
import math
import scipy.stats as sps
from sklearn.metrics import mean_squared_error, mean_absolute_error

sys.path.append(os.path.abspath('../../'))
from models.evaluate import evaluate
from models.data_loading import load_EOD_data, load_relation_data

In [2]:
seed = 123456789
random.seed(seed)
np.random.seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

In [3]:
class LSTMModel(nn.Module):
    def __init__(self, input_dim, hidden_units):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(input_dim, hidden_units, batch_first=True)
        self.fc = nn.Linear(hidden_units, 1)
        
        self._init_weights()

    def _init_weights(self):
        """Initialize weights using Xavier/Glorot uniform initialization."""
        for m in self.modules():
            if isinstance(m, (nn.Linear, nn.LSTM)):
                for name, param in m.named_parameters():
                    if 'bias' in name:
                        nn.init.zeros_(param)
                    elif 'weight' in name:
                        nn.init.xavier_uniform_(param)

    # Pass input through LSTM layer
    def forward(self, x):
        # x shape: (batch_size, seq_len, input_dim)
        
        # lstm_out shape: (batch_size, seq_len, hidden_units)
        lstm_out, _ = self.lstm(x)
        
        # Take the output of the last time step as the sequential embedding
        last_output = lstm_out[:, -1, :]
        
        # Pass the embedding through the fully-connected layer to get a prediction
        prediction = self.fc(last_output)
        
        # Apply Leaky ReLU activation)
        prediction = F.leaky_relu(prediction, 0.2)
        
        return prediction

In [4]:
class LSTM:
    def __init__(self, data_path, market_name, tickers_fname, parameters,
                 steps=1, epochs=50, batch_size=None, gpu=False):
        self.data_path = data_path
        self.market_name = market_name
        self.tickers_fname = tickers_fname
        self.tickers = np.genfromtxt(os.path.join(data_path, '..', tickers_fname),
                                     dtype=str, delimiter='\\t', skip_header=False)
        print('#tickers selected:', len(self.tickers))
        
        # Load the RAW prices for ground truth calculation in load_EOD_data
        raw_price_path = os.path.join(data_path, f'{market_name}_raw_prices.npy')
        self.raw_price_data = np.load(raw_price_path)
        if self.market_name == 'NASDAQ':
            self.raw_price_data = self.raw_price_data[:, :-1]
        print('Raw prices shape:', self.raw_price_data.shape)

        self.eod_data, self.mask_data, self.gt_data, _ = \
            load_EOD_data(data_path, market_name, self.tickers, self.raw_price_data, steps)
        
        self.parameters = copy.copy(parameters)
        self.steps = steps
        self.epochs = epochs
        self.batch_size = len(self.tickers) if batch_size is None else batch_size
        self.valid_index = 756
        self.test_index = 1008
        self.trade_dates = self.mask_data.shape[1]
        self.fea_dim = 5 # Number of features per day
        self.gpu = gpu

        self.device = torch.device('cuda' if gpu and torch.cuda.is_available() else 'cpu')
        print('device:', self.device)

    def get_batch(self, offset=None):
        if offset is None:
            offset = random.randrange(0, self.valid_index)
        seq_len = self.parameters['seq']
        mask_batch = self.mask_data[:, offset: offset + seq_len + self.steps]
        mask_batch = np.min(mask_batch, axis=1)

        # Handle masks for invalid prices
        base_price_batch = self.raw_price_data[:, offset + seq_len - 1]
        for i in range(len(mask_batch)):
            if base_price_batch[i] < 1e-8:
                mask_batch[i] = 0.0
        
        return self.eod_data[:, offset:offset + seq_len, :], \
               np.expand_dims(mask_batch, axis=1), \
               np.expand_dims(self.gt_data[:, offset + seq_len + self.steps - 1], axis=1)

    def compute_loss(self, pred, ground_truth, mask):
        """ Computes the simple MSE loss for the baseline LSTM. """
        loss = F.mse_loss(pred * mask, ground_truth * mask)
        return loss

    def train(self):
        model = LSTMModel(self.fea_dim, self.parameters['unit']).to(self.device)
        optimizer = optim.Adam(model.parameters(), lr=self.parameters['lr'])

        best_valid_perf = {'mse': np.inf}
        best_test_perf = {'mse': np.inf}
        best_valid_loss = np.inf

        for epoch in range(self.epochs):
            t1 = time()
            model.train()
            total_loss = 0.0

            batch_offsets = np.arange(0, self.valid_index)
            np.random.shuffle(batch_offsets)

            train_steps = self.valid_index - self.parameters['seq'] - self.steps + 1
            for j in range(train_steps):
                eod_batch, mask_batch, gt_batch = self.get_batch(batch_offsets[j])
                x = torch.tensor(eod_batch, dtype=torch.float32, device=self.device)
                mask = torch.tensor(mask_batch, dtype=torch.float32, device=self.device)
                gt = torch.tensor(gt_batch, dtype=torch.float32, device=self.device)

                optimizer.zero_grad()
                pred = model(x)
                loss = self.compute_loss(pred, gt, mask)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

                total_loss += loss.item()

            print(f"Epoch {epoch+1} | Train MSE: {total_loss / train_steps:.6f}")

            # Validation
            model.eval()
            with torch.no_grad():
                val_pred = np.zeros([len(self.tickers), self.test_index - self.valid_index])
                val_gt = np.zeros_like(val_pred)
                val_mask = np.zeros_like(val_pred)
                val_loss = 0.0

                val_steps = self.test_index - self.parameters['seq'] - self.steps + 1
                for offset in range(self.valid_index - self.parameters['seq'] - self.steps + 1, val_steps):
                    eod_batch, mask_batch, gt_batch = self.get_batch(offset)
                    x = torch.tensor(eod_batch, dtype=torch.float32, device=self.device)
                    mask = torch.tensor(mask_batch, dtype=torch.float32, device=self.device)
                    gt = torch.tensor(gt_batch, dtype=torch.float32, device=self.device)

                    pred = model(x)
                    loss = self.compute_loss(pred, gt, mask)
                    val_loss += loss.item()

                    idx = offset - (self.valid_index - self.parameters['seq'] - self.steps + 1)
                    val_pred[:, idx] = pred.squeeze().cpu().numpy()
                    val_gt[:, idx] = gt.squeeze().cpu().numpy()
                    val_mask[:, idx] = mask.squeeze().cpu().numpy()

                avg_val_loss = val_loss / (self.test_index - self.valid_index)
                print(f"Valid MSE: {avg_val_loss:.6f}")
                cur_valid_perf = evaluate(val_pred, val_gt, val_mask)
                print('\tValid performance:', cur_valid_perf)

                if avg_val_loss < best_valid_loss:
                    best_valid_loss = avg_val_loss
                    best_valid_perf = cur_valid_perf
                    print('Better valid loss:', best_valid_loss)
                    self.save_model(model, f'../../data/pretrain/pretrain/{self.market_name}_lstm_model.pt')
            
            print('Epoch:', epoch, 'Time: %.4f' % (time() - t1))
            
        print('\nBest Valid performance:', best_valid_perf)

    def save_model(self, model, path):
        torch.save({
            'model_state_dict': model.state_dict(),
            'input_dim': self.fea_dim,
            'hidden_units': self.parameters['unit']
        }, path)
        print(f"Model saved to {path}")

    def load_model(self, path):
        """
        Load a trained model from a file
        """
        checkpoint = torch.load(path, map_location=self.device)
        model = LSTMModel(
            input_dim=checkpoint['input_dim'],
            hidden_units=checkpoint['hidden_units']
        ).to(self.device)
        model.load_state_dict(checkpoint['model_state_dict'])
        model.eval()
        print(f"Model loaded from {path}")
        return model
    
    def predict(self, model, start=None):
        """
        Use the loaded model to make predictions on a given period (e.g., test set)
        """
        model.eval()
        with torch.no_grad():
            pred_dim = self.trade_dates - start
            test_pred = np.zeros([len(self.tickers), pred_dim])
            test_gt = np.zeros_like(test_pred)
            test_mask = np.zeros_like(test_pred)
    
            test_end_offset = self.trade_dates - self.parameters['seq'] - self.steps + 1
            for offset in range(start - self.parameters['seq'] - self.steps + 1, test_end_offset):
                # Get data for the current time step
                eod_batch, mask_batch, gt_batch = self.get_batch(offset)
                
                # Convert to tensors
                x = torch.tensor(eod_batch, dtype=torch.float32, device=self.device)
                mask = torch.tensor(mask_batch, dtype=torch.float32, device=self.device)
                gt = torch.tensor(gt_batch, dtype=torch.float32, device=self.device)
                
                # Get model prediction (which is the return ratio)
                prediction = model(x)
                
                # Store results
                idx = offset - (start - self.parameters['seq'] - self.steps + 1)
                test_pred[:, idx] = prediction.squeeze().cpu().numpy()
                test_gt[:, idx] = gt.squeeze().cpu().numpy()
                test_mask[:, idx] = mask.squeeze().cpu().numpy()

            performance = evaluate(test_pred, test_gt, test_mask)
                
            return (test_pred, test_gt, test_mask, performance)

### Training on NASDAQ

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [6]:
data_path = '../../data/2013-01-01'
market_name = 'NASDAQ'
tickers_fname = f"{market_name}_tickers_qualify_dr-0.98_min-5_smooth.csv"
parameters = {'seq': 16, 'unit': 64, 'lr': 0.001, 'alpha': 0.1}

In [7]:
lstm_baseline = LSTM(
    data_path=data_path,
    market_name=market_name,
    tickers_fname=tickers_fname,
    parameters=parameters,
    steps=1,
    epochs=50,
    batch_size=None,
    gpu=torch.cuda.is_available()
)
# Train the model
lstm_baseline.train()

#tickers selected: 1026
Raw prices shape: (1026, 1245)
single EOD data shape: (1245, 6)
device: cuda
Epoch 1 | Train MSE: 0.000629
Valid MSE: 0.000579
	Valid performance: {'mse': np.float64(0.0005833770201288075), 'mrrt': np.float64(0.009293122912234124), 'btl': np.float64(1.1040117020020261), 'btl5': np.float64(1.250563945077009), 'btl10': np.float64(1.2308974338731788)}
Better valid loss: 0.0005786364917950113
Model saved to ../../data/pretrain/pretrain/NASDAQ_lstm_model.pt
Epoch: 0 Time: 3.1528
Epoch 2 | Train MSE: 0.000462
Valid MSE: 0.000501
	Valid performance: {'mse': np.float64(0.000505486457268067), 'mrrt': np.float64(0.007864320029817264), 'btl': np.float64(0.6785862254182575), 'btl5': np.float64(1.0662656489119395), 'btl10': np.float64(1.1620375325255738)}
Better valid loss: 0.000501378865183575
Model saved to ../../data/pretrain/pretrain/NASDAQ_lstm_model.pt
Epoch: 1 Time: 2.4622
Epoch 3 | Train MSE: 0.000455
Valid MSE: 0.000517
	Valid performance: {'mse': np.float64(0.00052

### Training on NYSE

In [8]:
data_path = '../../data/2013-01-01'
market_name = 'NYSE'
tickers_fname = f"{market_name}_tickers_qualify_dr-0.98_min-5_smooth.csv"
parameters = {'seq': 8, 'unit': 32, 'lr': 0.001, 'alpha': 10}

In [9]:
lstm_baseline = LSTM(
    data_path=data_path,
    market_name=market_name,
    tickers_fname=tickers_fname,
    parameters=parameters,
    steps=1,
    epochs=50,
    batch_size=None,
    gpu=torch.cuda.is_available()
)
# Train the model
lstm_baseline.train()

#tickers selected: 1737
Raw prices shape: (1737, 1245)
single EOD data shape: (1245, 6)
device: cuda
Epoch 1 | Train MSE: 0.000320
Valid MSE: 0.000397
	Valid performance: {'mse': np.float64(0.00039742283783794115), 'mrrt': np.float64(0.0037277463728801496), 'btl': np.float64(0.6736324788798811), 'btl5': np.float64(1.0743796947614566), 'btl10': np.float64(1.0781256562090902)}
Better valid loss: 0.0003968045500172709
Model saved to ../../data/pretrain/pretrain/NYSE_lstm_model.pt
Epoch: 0 Time: 6.3892
Epoch 2 | Train MSE: 0.000283
Valid MSE: 0.000372
	Valid performance: {'mse': np.float64(0.000372925111173065), 'mrrt': np.float64(0.00419380536920513), 'btl': np.float64(0.4880384236721511), 'btl5': np.float64(0.9834738121391644), 'btl10': np.float64(1.0493254200034015)}
Better valid loss: 0.0003723449337398744
Model saved to ../../data/pretrain/pretrain/NYSE_lstm_model.pt
Epoch: 1 Time: 2.8730
Epoch 3 | Train MSE: 0.000277
Valid MSE: 0.000388
	Valid performance: {'mse': np.float64(0.000388